In [5]:
import torch
import numpy as np
import pandas as pd
import pickle
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
from scipy.stats import entropy
from scipy.special import kl_div
import warnings
warnings.filterwarnings('ignore')

# Load model
tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")
model.eval()
tokenizer.pad_token = tokenizer.eos_token

# Load saved composite model
with open('results/best_model.pkl', 'rb') as f:
    best_model = pickle.load(f)
with open('results/best_scaler.pkl', 'rb') as f:
    best_scaler = pickle.load(f)
with open('results/best_features.pkl', 'rb') as f:
    best_features = pickle.load(f)

print("=" * 55)
print("  NLP Assignment — Track A Live Demo Pipeline")
print("  CS F429 | BITS Pilani Dubai | Semester II 2025-26")
print("=" * 55)
print(f"\nModel: GPT-2")
print(f"Composite features: {best_features}")
print("\nReady to accept input.")

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 6159.59it/s]


  NLP Assignment — Track A Live Demo Pipeline
  CS F429 | BITS Pilani Dubai | Semester II 2025-26

Model: GPT-2
Composite features: ['entropy_only', 'information_gain', 'kl_divergence', 'confidence_drop']

Ready to accept input.


In [6]:
def run_demo_pipeline(question, context, answer):
    """
    Full end-to-end hallucination detection pipeline.
    Takes a question, retrieved context, and model answer.
    Returns token-level metric scores and hallucination prediction.
    """
    print("\n" + "=" * 55)
    print("INPUT")
    print("=" * 55)
    print(f"Question:  {question[:100]}")
    print(f"Context:   {context[:150]}...")
    print(f"Answer:    {answer[:150]}...")

    # Step 1: Get token probabilities with and without context
    def get_probs(text, ctx=None, max_length=200):
        input_text = f"Context: {ctx[:400]}\n\nAnswer: {text}" if ctx else text
        inputs = tokenizer(input_text, return_tensors="pt",
                          max_length=max_length, truncation=True)
        with torch.no_grad():
            outputs = model(**inputs)
            probs = torch.softmax(outputs.logits, dim=-1)
        return probs[0].numpy()

    probs_no_ctx = get_probs(answer)
    probs_with_ctx = get_probs(answer, ctx=context)
    min_len = min(len(probs_no_ctx), len(probs_with_ctx))
    probs_no_ctx = probs_no_ctx[:min_len]
    probs_with_ctx = probs_with_ctx[:min_len]

    # Step 2: Compute entropy at each token position
    entropy_no_ctx = np.array([entropy(p) for p in probs_no_ctx])
    entropy_with_ctx = np.array([entropy(p) for p in probs_with_ctx])

    # Step 3: Compute Information Gain per token
    # IG = entropy without context - entropy with context
    # High IG = context helped = faithful
    # Low IG = context didn't help = possible hallucination
    ig_per_token = entropy_no_ctx - entropy_with_ctx

    # Step 4: Compute KL Divergence per token
    # KL measures how much the probability distribution shifted
    # when context was added
    kl_per_token = []
    for p_no, p_with in zip(probs_no_ctx, probs_with_ctx):
        p_no = p_no + 1e-10; p_no = p_no / p_no.sum()
        p_with = p_with + 1e-10; p_with = p_with / p_with.sum()
        kl_per_token.append(float(np.sum(kl_div(p_no, p_with))))

    # Step 5: Compute Confidence Drop per token
    conf_no_ctx = np.max(probs_no_ctx, axis=1)
    conf_with_ctx = np.max(probs_with_ctx, axis=1)
    conf_drop_per_token = conf_no_ctx - conf_with_ctx

    # Step 6: Aggregate to sequence level
    metrics = {
        'entropy_only':     float(np.mean(entropy_no_ctx)),
        'information_gain': float(np.mean(ig_per_token)),
        'kl_divergence':    float(np.mean(kl_per_token)),
        'confidence_drop':  float(np.mean(conf_drop_per_token)),
    }

    # Step 7: Compute composite hallucination score
    X = np.array([[metrics[f] for f in best_features]])
    X_scaled = best_scaler.transform(X)
    hallucination_prob = best_model.predict_proba(X_scaled)[0][1]

    print("\n" + "=" * 55)
    print("TOKEN-LEVEL METRIC SCORES")
    print("=" * 55)
    print(f"  Entropy Only:      {metrics['entropy_only']:.4f}")
    print(f"  Information Gain:  {metrics['information_gain']:.4f}  (higher = more faithful)")
    print(f"  KL Divergence:     {metrics['kl_divergence']:.4f}  (higher = more context shift)")
    print(f"  Confidence Drop:   {metrics['confidence_drop']:.4f}")

    print("\n" + "=" * 55)
    print("TOKEN-LEVEL IG SCORES (first 10 tokens)")
    print("=" * 55)
    tokens = tokenizer.encode(answer)[:10]
    for i, (tok, ig) in enumerate(zip(tokens, ig_per_token[:10])):
        token_str = tokenizer.decode([tok])
        bar = "█" * int(abs(ig) * 2)
        print(f"  Token {i+1:2d} '{token_str:<12}' IG={ig:+.4f} {bar}")

    print("\n" + "=" * 55)
    print("HALLUCINATION PREDICTION")
    print("=" * 55)
    print(f"  Composite Score:      {hallucination_prob:.4f}")
    if hallucination_prob > 0.5:
        print(f"  Prediction:           ⚠️  HALLUCINATED (confidence: {hallucination_prob*100:.1f}%)")
    else:
        print(f"  Prediction:           ✅  FAITHFUL (confidence: {(1-hallucination_prob)*100:.1f}%)")
    print("=" * 55)

    return metrics, hallucination_prob

# Test with a known hallucinated example
question = "When did Anne Frank die?"
context = "Anne Frank died in the Bergen-Belsen concentration camp in February 1945, likely around February 12."
answer = "Anne Frank died on February 7, 2022, according to the Anne Frank House."

metrics, score = run_demo_pipeline(question, context, answer)


INPUT
Question:  When did Anne Frank die?
Context:   Anne Frank died in the Bergen-Belsen concentration camp in February 1945, likely around February 12....
Answer:    Anne Frank died on February 7, 2022, according to the Anne Frank House....

TOKEN-LEVEL METRIC SCORES
  Entropy Only:      3.8143
  Information Gain:  0.3003  (higher = more faithful)
  KL Divergence:     8.6629  (higher = more context shift)
  Confidence Drop:   -0.1326

TOKEN-LEVEL IG SCORES (first 10 tokens)
  Token  1 'Anne        ' IG=+0.3374 
  Token  2 ' Frank      ' IG=-2.1751 ████
  Token  3 ' died       ' IG=-3.3829 ██████
  Token  4 ' on         ' IG=-1.7795 ███
  Token  5 ' February   ' IG=+0.0390 
  Token  6 ' 7          ' IG=-3.4632 ██████
  Token  7 ',           ' IG=-1.8097 ███
  Token  8 ' 2022       ' IG=+1.1461 ██
  Token  9 ',           ' IG=+0.7928 █
  Token 10 ' according  ' IG=-0.0432 

HALLUCINATION PREDICTION
  Composite Score:      0.5679
  Prediction:           ⚠️  HALLUCINATED (confidence: 56

In [7]:
question = "When did Anne Frank die?"

context = """Anne Frank died in the Bergen-Belsen 
concentration camp in February 1945, 
likely around February 12."""

answer = "Anne Frank died on February 7, 2022, according to the Anne Frank House."

In [8]:
question = "When did Anne Frank die?"

context = """Anne Frank died in the Bergen-Belsen 
concentration camp in February 1945, 
likely around February 12."""

answer = "Anne Frank died on February 7, 2022, according to the Anne Frank House."